In [1]:
import pickle
import os
import pandas as pd
import numpy as np


In [2]:
train_file = "/kaggle/input/fii-nn-2025-homework-3/extended_mnist_train.pkl"
test_file = "/kaggle/input/fii-nn-2025-homework-3/extended_mnist_test.pkl"

with open(train_file, "rb") as fp:
    train = pickle.load(fp)

with open(test_file, "rb") as fp:
    test = pickle.load(fp)

In [3]:
train_data = []
train_labels = []
for image, label in train:
    train_data.append(image.flatten())
    train_labels.append(label)


In [4]:
test_data = []
for image, label in test:
    test_data.append(image.flatten())


In [5]:
X = np.array(train_data, dtype=np.float32)
y = np.array(train_labels, dtype=np.int64)
X_test = np.array(test_data, dtype=np.float32)
print("Train label stats:", y.min(), y.max(), np.unique(y)[:12])
X=X/255.0
X_test=X_test/255.0

mean_img=X.mean(axis=0, keepdims=True).astype(np.float32)
X=X -mean_img
X_test_centered=X_test -mean_img
num_classes=10
rng= np.random.default_rng(12)
idx= rng.permutation(len(X))
val_ratio= 0.1667
val_size= int(len(X)*val_ratio)
val_idx= idx[:val_size]
train_idx= idx[val_size:]
X_train, y_train= X[train_idx], y[train_idx]
X_val, y_val= X[val_idx], y[val_idx]

Train label stats: 0 9 [0 1 2 3 4 5 6 7 8 9]


In [6]:
def softmax(z):
    # pt a nu creste prea mult exponentialul
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

def one_hot(y, k):
    Y = np.zeros((y.size, k), dtype=np.float32)
    Y[np.arange(y.size), y]= 1.0
    return Y

def cross_entropy(Y, P):
    eps= 1e-12
    return -np.mean(np.sum(Y*np.log(P+eps), axis=1))

def accuracy(logits, y_true):
    preds= logits.argmax(axis=1)
    return(preds== y_true).mean()

def relu_grad(x): return (x > 0).astype(np.float32)

def init_params(in_dim, hidden, out_dim, seed=42):
    rng= np.random.default_rng(seed)
    W1= rng.normal(0, np.sqrt(2/in_dim), (in_dim, hidden)).astype(np.float32)#he
    b1= np.zeros((1, hidden), dtype=np.float32)
    W2= rng.normal(0, np.sqrt(1/hidden), (hidden, out_dim)).astype(np.float32)#xavier
    b2= np.zeros((1, out_dim), dtype=np.float32)
    return W1, b1, W2, b2


In [7]:
def forward(X, W1, b1, W2, b2, dropout_p=0.0, rng=None, train_mode=True):
    Z1=X @W1+b1
    A1= np.maximum(Z1, 0.0)
    mask1= None
    if train_mode and dropout_p > 0.0:
        keep_p= 1.0 - dropout_p
        mask1= (rng.random(A1.shape)< keep_p).astype(np.float32)/ keep_p
        A1= A1* mask1
    Z2= A1 @W2+b2
    return Z1,A1,Z2,mask1

def backward(X, Y, Z1, A1, Z2, W1, W2, l2, mask1=None):
    B= X.shape[0]
    P= softmax(Z2)
    loss= cross_entropy(Y, P)+0.5*l2*(np.sum(W1*W1)+np.sum(W2*W2))

    dZ2= (P- Y) / B
    dW2= A1.T @ dZ2 + l2*W2
    db2=np.sum(dZ2, axis=0, keepdims=True)

    dA1= dZ2 @ W2.T
    dZ1=dA1 * relu_grad(Z1)
    if mask1 is not None:
        dZ1*=mask1

    dW1= X.T @ dZ1 + l2*W1
    db1=np.sum(dZ1, axis=0, keepdims=True)
    return loss,dW1, db1, dW2, db2

def train_mlp(X_train, y_train, X_val, y_val,lr=0.2, epochs=35, batch_size=256, l2=5e-4, seed=12,
        dropout_p=0.1, plateau_patience=5,lr_decay_every=10, lr_decay_factor=0.5):
    rng= np.random.default_rng(seed)
    m, n= X_train.shape
    k= int(y_train.max()) + 1
    W1, b1, W2, b2= init_params(n, 100, k, seed)
    Y_train= one_hot(y_train, k)
    Y_val= one_hot(y_val,k)

    idx= np.arange(m)
    best_val_loss= np.inf
    best_params= None
    stall= 0

    for ep in range(1, epochs+1):
        rng.shuffle(idx)
        X_train, Y_train, y_train= X_train[idx], Y_train[idx], y_train[idx]

        total_loss, seen= 0.0, 0
        for start in range(0, m, batch_size):
            end= min(start + batch_size, m)
            Xb, Yb= X_train[start:end], Y_train[start:end]

            Z1, A1, Z2, mask1= forward(Xb, W1, b1, W2, b2, dropout_p, rng, train_mode=True)
            loss, dW1, db1, dW2, db2= backward(Xb, Yb, Z1, A1, Z2, W1, W2, l2, mask1)

            W1-= lr*dW1; 
            b1-= lr*db1
            W2-=lr*dW2; 
            b2-= lr*db2

            bs= end - start
            total_loss+= loss * bs
            seen+= bs 

        if ep%lr_decay_every == 0:
            lr*= lr_decay_factor

        _, _, Z2train, _= forward(X_train, W1, b1, W2, b2, dropout_p=0.0, rng=None, train_mode=False)
        _, _, Z2val, _= forward(X_val,   W1, b1, W2, b2, dropout_p=0.0, rng=None, train_mode=False)
        train_acc= accuracy(Z2train, y_train)
        val_acc= accuracy(Z2val, y_val)

        P_val= softmax(Z2val)
        val_loss= cross_entropy(Y_val, P_val) + 0.5*l2*(np.sum(W1*W1) + np.sum(W2*W2))

        print(f"Epoch {ep:02d} | loss={total_loss/seen:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | val_loss={val_loss:.4f}")

        if val_loss+1e-6 < best_val_loss:
            best_val_loss= val_loss
            best_params= (W1.copy(), b1.copy(), W2.copy(), b2.copy())
            stall= 0
        else:
            stall+= 1
            if stall>= plateau_patience:
                lr*= lr_decay_factor
                stall= 0

    if best_params is not None:
        W1, b1, W2, b2= best_params
    return W1, b1, W2, b2

In [8]:
W1, b1, W2, b2= train_mlp( X_train, y_train, X_val, y_val, lr=3.2, epochs=50, batch_size=256,
    l2=3e-4, seed=12, dropout_p=0.2, plateau_patience=3, lr_decay_every=10, lr_decay_factor=0.4 )

def predict(X, W1, b1, W2, b2):
    _, _, Z2, _= forward(X, W1, b1, W2, b2, dropout_p=0.0, rng=None, train_mode=False)
    return np.argmax(softmax(Z2), axis=1)


predictions= predict(X_test_centered, W1, b1, W2, b2).astype(int)
df= pd.DataFrame({"ID": np.arange(len(predictions)), "target": predictions.astype(int)})
df.to_csv("submission.csv", index=False)


Epoch 01 | loss=0.3728 | train_acc=0.9218 | val_acc=0.9135 | val_loss=0.3097
Epoch 02 | loss=0.2077 | train_acc=0.9716 | val_acc=0.9588 | val_loss=0.1938
Epoch 03 | loss=0.1814 | train_acc=0.9704 | val_acc=0.9582 | val_loss=0.1959
Epoch 04 | loss=0.1681 | train_acc=0.9784 | val_acc=0.9653 | val_loss=0.1758
Epoch 05 | loss=0.1606 | train_acc=0.9798 | val_acc=0.9689 | val_loss=0.1719
Epoch 06 | loss=0.1541 | train_acc=0.9827 | val_acc=0.9695 | val_loss=0.1673
Epoch 07 | loss=0.1532 | train_acc=0.9867 | val_acc=0.9716 | val_loss=0.1567
Epoch 08 | loss=0.1488 | train_acc=0.9753 | val_acc=0.9595 | val_loss=0.1925
Epoch 09 | loss=0.1495 | train_acc=0.9854 | val_acc=0.9707 | val_loss=0.1611
Epoch 10 | loss=0.1459 | train_acc=0.9876 | val_acc=0.9719 | val_loss=0.1590
Epoch 11 | loss=0.1241 | train_acc=0.9933 | val_acc=0.9787 | val_loss=0.1351
Epoch 12 | loss=0.1160 | train_acc=0.9942 | val_acc=0.9789 | val_loss=0.1336
Epoch 13 | loss=0.1130 | train_acc=0.9952 | val_acc=0.9791 | val_loss=0.1303

In [9]:
# This is how you prepare a submission for the competition
# predictions_csv = {
#     "ID": [],
#     "target": [],
# }

# for i, label in enumerate(predictions):
#     predictions_csv["ID"].append(i)
#     predictions_csv["target"].append(label)

# df = pd.DataFrame(predictions_csv)
# df.to_csv("submission.csv", index=False)